# Use case #1 — Debug an error with an LLM
## Lexical Search (BM25)


## Index the corpus

**We index `##` sections, not whole documents.** A doc that mentions our error once is mostly noise; the section *about* the error is the answer. The retrieval unit is also the **prompt** unit — whatever wins here is what the LLM actually reads.

Flip `SPLIT` to `"fixed"` and the headings are stripped and the prose cut into 800-char chunks instead — the fallback for Confluence exports, PDFs, or anything without reliable structure. Nothing downstream changes, and the ranking barely moves.

Tokenization is where lexical search is really won or lost, so we emit **both** the exact compound and its parts. Otherwise `CONFIGURATION_IS_MISSING` shatters into `configuration` — a word that appears in half the corpus.

In [1]:
import pandas as pd
from acme import load_chunks, tokenize, bm25_index, show

SPLIT = "sections"   # "sections" = split on ## headers | "fixed" = 800-char chunks, no structure

chunks = load_chunks(split=SPLIT)
print(f"{len({c.filename for c in chunks})} documents  ->  {len(chunks)} chunks indexed ({SPLIT})\n")

for s in ["CONFIGURATION_IS_MISSING", "get_tool_config", "sapir.startup.ConfigLoader"]:
    print(f"  {s:28} -> {tokenize(s)}")

16 documents  ->  71 chunks indexed (sections)

  CONFIGURATION_IS_MISSING     -> ['configuration_is_missing', 'configuration', 'is', 'missing']
  get_tool_config              -> ['get_tool_config', 'get', 'tool', 'config']
  sapir.startup.ConfigLoader   -> ['sapir.startup.configloader', 'sapir', 'startup', 'config', 'loader']


## Search

In [2]:
query = "We saw the following Acme error in the logs: CONFIGURATION_IS_MISSING, what should I do?"

bm25 = bm25_index(chunks)
show(chunks, bm25.get_scores(tokenize(query)), top_k=6)

  17.419   sapir-config-errors-legacy.md   § CONFIGURATION_IS_MISSING  ⚠️ superseded
  14.194   sapir-config-errors.md          § CONFIGURATION_IS_MISSING
  12.786   runbook-startup-failures.md     § If Sapir is the failed component
  10.316   sapir-conf-migration-2025.md    § Errors during migration  ⚠️ superseded
  10.250   sapir-config-errors-legacy.md   § Sapir — Configuration Errors (legacy)  ⚠️ superseded
   9.261   sapir-config-errors.md          § Quick reference


**BM25 nailed the topic.** Out of 71 sections it surfaced Sapir configuration errors instantly. And note who *didn't* show up: `onboarding-acme.md` says *Acme* more than any other doc, but "Acme" appears in **every** document and so carries no ranking power at all. The one rare term decides everything — that's IDF, without saying "IDF".

**And the top hit is a superseded 2024 page**, whose fix (`sapir --reload-config`) was deleted in the 2025 migration. Another stale doc sits a few rows below. BM25 isn't broken — the legacy page repeats our exact term more densely, so by every lexical measure it genuinely *is* the better match.

Paste that into an LLM and it confidently tells your operator to run a flag that no longer exists.

In [3]:
# the signal we needed: parsed, available, and never part of the score
pd.DataFrame(sorted({(c.filename, c.meta.get("status", "—")[:30], c.meta.get("last_updated", ""))
                     for c in chunks}), columns=["file", "status", "last_updated"])

,file,status,last_updated
0,incident-2026-03-config-outage.md,current,2026-03-12
1,kesh-auth-errors.md,current,2026-07-01
2,lomi-gateway-errors.md,current,2026-06-02
3,lomi-rate-limiting.md,current,2025-11-18
4,nugat-memory-tuning.md,current,2026-05-02
5,nugat-write-path-errors.md,current,2026-03-30
6,onboarding-acme.md,current,2026-01-10
7,acme-architecture-overview.md,current,2026-06-30
8,acme-config-reference.md,current,2026-07-15
9,runbook-startup-failures.md,current,2026-04-05


## Takeaway

**Lexical search is unbeatable at what it does and blind to everything else.** It answers *"which text looks most like this text?"* — never *"which answer is still true?"*

| Next method | The question BM25 can't answer |
| --- | --- |
| **Semantic** (`02`) | *same incident, described instead of quoted* — the rare string is gone |
| **Structured** (`03`) | *"...only from docs that aren't superseded"* — freshness is a filter, not a score |
| **Graph** (`04`) | *"who should I talk with, and what else breaks?"* — the answer isn't in any single section |
